# Bab 8 - Learning Signal and Ignoring Noise: Regularization and Batching

**Nama:** Nanda Pratama ITM  
**NIM:** 101032330212  
**Kelas:** TK 47 01


> Notebook ini merupakan reproduksi edukatif dan pendalaman teori berdasarkan *Grokking Deep Learning* (Andrew W. Trask, Manning, 2019). Kode inti diadaptasi dari buku dan repositori pendamping resmi; perubahan kompatibilitas dijelaskan dalam dokumentasi.


## Capaian pembelajaran

Setelah menyelesaikan notebook ini, pembaca diharapkan mampu menjelaskan konsep utama bab, menghubungkannya dengan operasi NumPy, menjalankan contoh secara berurutan, dan menyebutkan batas interpretasinya.


## MNIST dan generalization

MNIST menyediakan gambar digit dan label. Pemisahan training-test harus dipertahankan: model belajar hanya dari training, sementara test dipakai untuk estimasi akhir. Akurasi training tinggi dapat bersamaan dengan akurasi test yang buruk.


## Overfitting

Overfitting terjadi ketika model menangkap pola khusus atau noise training yang tidak berulang pada data baru. Kapasitas besar, data sedikit, dan pelatihan terlalu lama meningkatkan risikonya.


## Early stopping

Early stopping memantau loss atau metrik validasi dan menghentikan pelatihan ketika perbaikan berhenti. Test set tidak boleh dipakai untuk memilih kapan berhenti karena akan membocorkan informasi.


## Dropout

Dropout menonaktifkan unit secara acak saat training dan menskalakan aktivasi agar ekspektasinya konsisten. Ini menyerupai ensemble banyak subnet. Saat evaluasi, dropout dimatikan.


## Mini-batch

Mini-batch merata-ratakan gradien beberapa contoh. Ukuran batch memengaruhi memori, kecepatan, dan noise gradien; tidak ada satu ukuran yang selalu optimal.


## Reproduksi kode dan eksperimen

Jalankan sel dari atas ke bawah. `MODE_CEPAT=True` menjaga waktu eksekusi singkat; ubah ke `False` untuk konfigurasi yang lebih panjang.


In [1]:
from pathlib import Path
import numpy as np

np.set_printoptions(precision=4, suppress=True)
np.random.seed(1)
DATA_DIR = (Path.cwd().parent / "data").resolve()
MODE_CEPAT = True  # ubah ke False untuk eksperimen lebih panjang
assert DATA_DIR.is_dir(), f"Folder data tidak ditemukan: {DATA_DIR}"
print("Data:", DATA_DIR)
print("Mode cepat:", MODE_CEPAT)

def sigmoid(x):
    x = np.clip(x, -60, 60)
    return 1 / (1 + np.exp(-x))

def softmax(x):
    x = np.asarray(x)
    shifted = x - np.max(x, axis=-1, keepdims=True)
    exp = np.exp(shifted)
    return exp / exp.sum(axis=-1, keepdims=True)


Data: D:\Kuliah\DL\UPLOAD_GITHUB\Grokking-Deep-Learning\data
Mode cepat: True


In [2]:
with np.load(DATA_DIR / "mnist.npz") as mnist:
    x_train, y_train = mnist["x_train"], mnist["y_train"]
    x_test, y_test = mnist["x_test"], mnist["y_test"]

n_train = 1200 if MODE_CEPAT else 6000
n_test = 500 if MODE_CEPAT else 2000
x_train = x_train[:n_train].reshape(n_train, -1).astype(float) / 255
y_train = y_train[:n_train]
x_test = x_test[:n_test].reshape(n_test, -1).astype(float) / 255
y_test = y_test[:n_test]
y_onehot = np.eye(10)[y_train]
print(x_train.shape, x_test.shape)


(1200, 784) (500, 784)


In [3]:
rng = np.random.default_rng(8)
hidden, alpha, epochs, batch_size = 64, 0.08, (8 if MODE_CEPAT else 30), 100
w01 = rng.normal(0, 0.03, (784, hidden))
w12 = rng.normal(0, 0.03, (hidden, 10))

def accuracy(x, y):
    return np.mean(np.argmax(np.maximum(0, x @ w01) @ w12, axis=1) == y)

history = []
for epoch in range(epochs):
    order = rng.permutation(len(x_train))
    for start in range(0, len(order), batch_size):
        idx = order[start:start+batch_size]
        l0 = x_train[idx]
        z1 = l0 @ w01
        l1 = np.maximum(0, z1)
        keep = (rng.random(l1.shape) > 0.5) * 2.0
        l1_drop = l1 * keep
        l2 = l1_drop @ w12
        delta2 = (l2 - y_onehot[idx]) / len(idx)
        delta1 = (delta2 @ w12.T) * (z1 > 0) * keep
        w12 -= alpha * l1_drop.T @ delta2
        w01 -= alpha * l0.T @ delta1
    history.append((accuracy(x_train, y_train), accuracy(x_test, y_test)))
print("train/test accuracy:", np.round(history[-1], 4))


train/test accuracy: [0.7508 0.668 ]


## Interpretasi dan kesimpulan

Hasil harus dibaca bersama asumsi, konfigurasi, seed, serta data yang digunakan. Keberhasilan menjalankan kode tidak membuktikan generalisasi, kausalitas, keamanan, atau kelayakan deployment. Bandingkan hasil dengan teori di atas dan jelaskan setiap perbedaan sebelum mengubah konfigurasi.
